### Wczytanie danych (i bibliotek)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

# 1. Eksploracyjna analiza danych

### Wczytanie i zrozumienie danych

In [ ]:
df = pd.read_csv('../data/Hotel Reservations.csv')
print(f"Liczba wierszy: {df.shape[0]}, liczba kolumn: {df.shape[1]}")
df.head()

### Typy danych i zmiennych

In [ ]:
df.info()

In [ ]:
print("Zmienne numeryczne")
df.describe()

In [ ]:
print("Zmienne opisowe")
df.describe(include='object')

In [ ]:
df.market_segment_type.unique()

### Sprawdzenie brakujących danych

In [ ]:
print("Liczba brakujących wartości")
df.isnull().sum()

In [ ]:
print("Liczba duplikatów")
df.duplicated().sum()

In [ ]:
df['booking_status'].value_counts(normalize=True) # Można tym sprawdzić sobie stosunek cancelled/not cancelled

### Hipotezy

Analizująć kolejne kolumny z ramki danych możemy postawić następujące hipotezy:
- im wcześniej robiono rezerwację, tym większa szansa anulacji (lead_time)
- powracający goście mogą rzadziej anulować rezerwacje (repeated_guest)
- segmenty kanałów sprzedaży mogą różnie wpływać na szansę anulacji, np.rezerwacje firmowe mogą być rzadziej odwoływane (market_segment_type)


In [ ]:
plt.figure(figsize=(7,4))
sns.boxplot(data=df, x='booking_status', y='lead_time')
plt.title('Lead time vs booking_status')
plt.show()

Im wcześniej robiono rezerwację, tym większa szansa na odwołanie.

In [ ]:
pd.crosstab(df["repeated_guest"], df["booking_status"], normalize="index") * 100

Powracający goście mają o wiele rzadziej odwołują rezerwacje.

In [ ]:
mseg = pd.crosstab(df["market_segment_type"], df["booking_status"], normalize="index") * 100
mseg = mseg.round(2)
display(mseg)
mseg.plot(kind="bar", stacked=True, figsize=(8,4))
plt.title("booking_status w grupach market_segment_type (w %)")
plt.xlabel("market_segment_type")
plt.ylabel("Procent")
plt.legend(title="booking_status")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

Rezerwacje robione online są najbardziej podatne na anulowanie. Najmniej odwołanych rezerwacji było w segmencie "Compemantary".

### Analiza rozkładów i zależności

In [ ]:
target_col = 'booking_status'
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
cat_cols.remove(target_col)
print('Numeryczne:', num_cols)
print("Kategoryczne:", cat_cols)

In [ ]:
df.no_of_previous_bookings_not_canceled

### Najważniejsze zmienne numeryczne

In [ ]:
important_num = [
    'lead_time',
    'repeated_guest',
    'no_of_previous_cancellations',
    'no_of_previous_bookings_not_canceled',
    'avg_price_per_room',
    'no_of_week_nights',
    'no_of_weekend_nights',
    'no_of_special_requests'
]

fig, ax = plt.subplots(4, 2, figsize=(14, 14))

for i, col in enumerate(important_num):
    sns.histplot(df[col], bins=30, kde=True, ax=ax[i//2, i%2])
    ax[i//2, i%2].set_title(f'Rozkład: {col}')
    ax[i//2, i%2].set_xlabel(col)
    ax[i//2, i%2].set_ylabel('Liczba obserwacji')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(4, 2, figsize=(14, 14))

for i, col in enumerate(important_num):
    sns.boxplot(x=df[col], ax=ax[i//2, i%2])
    ax[i//2, i%2].set_title(f'Boxplot: {col}')
    ax[i//2, i%2].set_xlabel(col)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(5,4))
sns.countplot(data=df, x='repeated_guest')
plt.title("Rozkład repeated_guest")
plt.show() 
# jest to zmienna binarna, więc lepiej ją widać na wykresie słupkowym

In [ ]:
corr = df[important_num].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Macierz korelacji cech numerycznych')
plt.tight_layout()
plt.show()

Widać, że najbardziej skorelowane ze sobą cechy to:
- repeated_guest i no_of_previous_bookings_not_canceled
- repeated_guest i no_of_previous_cancellations
- no_of_previous_cancellations i no_of_previous_bookings_not_canceled
Jest to zgodne z intuicją - poracający goście częściej posiadają historię wcześniejszych rezerwacji. Pozostałe korelacje nie wskazują na istotane zależności liniowe.


### Najważniejsze zmienne kategoryczne

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4))

important_cat = [
    "market_segment_type",
    "room_type_reserved",
    "type_of_meal_plan"
]

for i, col in enumerate(important_cat):
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order, ax=ax[i])
    ax[i].set_title(f"Rozkład: {col}")
    ax[i].set_xlabel(col)
    ax[i].set_ylabel("Liczba")
    ax[i].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

### Badanie niuansu w postaci outlierów ceny pokoju wynoszącej 0

In [ ]:
free_rooms = df[df['avg_price_per_room'] == 0]

print(f"Darmowych pokoi: {len(free_rooms)} ({len(free_rooms)/len(df)*100:.2f}% całego zbioru)")

cols_to_plot = ['market_segment_type', 'repeated_guest', 'booking_status']

fig, ax = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle("Analiza darmowych rezerwacji (avg_price_per_room = 0)", fontsize=14, y=1.05)

for i, col in enumerate(cols_to_plot):
    sns.countplot(data=free_rooms, x=col, ax=ax[i], order=free_rooms[col].value_counts().index, palette='viridis')
    ax[i].set_title(col)

plt.tight_layout()
plt.show()

In [ ]:
free_rooms = df[df['avg_price_per_room'] == 0]

print(free_rooms['repeated_guest'].value_counts())

print(free_rooms['booking_status'].value_counts())

counts = free_rooms['booking_status'].value_counts()
cancelled = counts.get('Canceled', 0)
not_cancelled = counts.get('Not_Canceled', 0)
percentage = cancelled / (cancelled + not_cancelled) * 100

print(f"\nProcent anulowanych: {percentage:.2f}%")

Główną obserwacją jest to, że pokoje te mają marginalny odsetek anulacji, bardzo rzadko kiedy pobyt w takich pokojach nie odbywa się.

### Wnioski

Na podstawie powyższych wykresów, możemy zauważyć, że:
- większość klientów rezerwuje do ok. 3 miesięcy z wyprzedzeniem, ale część rezerwuje bardzo wcześnie, nawet ponad rok wcześniej (dużo wartości odstających)
- więcej wartości odstających przy braku anulowania rezerwacji, niż przy anulowaniu rezerwacji, czyli większość klientów nie ma historii anulacji
- cena pokojów ma bardzo dużo outlierów
- większość pobytów jest krótkich, zwłaszcza weekendowych
- większość gości nie zgłasza specjalnych próśb
- najwięcej rezerwacji było robionych online
- najczęściej był rezerwowany pokój nr 1
- najczęściej był wybierany posiłek nr 1
- w przypadku darmowych pokojów, częstotliwość anulowania rezerwacji wynosiła ok 1%